# T1.1 — Análise Exploratória do PaySim

Notebook da Pessoa 1. Carrega o PaySim real (se disponível) ou uma amostra
sintética determinística com o mesmo contrato, inspeciona a distribuição das
transações, o desbalanceamento de classes e os sinais associados à fraude.

> Execute a partir da raiz do repositório ou de `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from data.loader import dataset_info, load_paysim

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [ ]:
df = load_paysim(rows=60_000, seed=42)
info = dataset_info(df)
print(info)
df.head()

## 1. Qualidade e contrato dos dados

Verificamos tipos, valores ausentes e duplicidades antes de qualquer feature
engineering.

In [ ]:
print(df.dtypes)
print("\nValores ausentes:")
print(df.isna().sum())
print(f"\nDuplicadas: {df.duplicated().sum()}")
df.describe(include="all").T

## 2. Desbalanceamento de classes

A fraude é rara — por isso o estudo prioriza **PR-AUC**, **recall** e **F1**
em vez de acurácia.

In [ ]:
counts = df["is_fraud"].value_counts().rename({False: "legítima", True: "fraude"})
print(counts)
print(f"Taxa de fraude: {df['is_fraud'].mean():.3%}")

ax = counts.plot(kind="bar", color=["#2563eb", "#dc2626"], figsize=(5, 3))
ax.set_title("Distribuição das classes")
ax.set_ylabel("Transações")
plt.show()

## 3. Fraude por tipo de transação

No PaySim a fraude aparece apenas em `TRANSFER` e `CASH_OUT`.

In [ ]:
by_type = (
    df.groupby("type")["is_fraud"]
    .agg(transacoes="size", fraudes="sum", taxa="mean")
    .sort_values("transacoes", ascending=False)
)
by_type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["type"].value_counts().plot(kind="bar", ax=axes[0], color="#2563eb")
axes[0].set_title("Volume por tipo")
axes[0].set_ylabel("Transações")

by_type["taxa"].plot(kind="bar", ax=axes[1], color="#dc2626")
axes[1].set_title("Taxa de fraude por tipo")
axes[1].set_ylabel("Proporção")

plt.tight_layout()
plt.show()

## 4. Valores e saldos

A escala é fortemente assimétrica; usamos escala log nos gráficos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(np.log1p(df["amount"]), bins=60, ax=axes[0], color="#2563eb")
axes[0].set_title("log(1 + amount)")
axes[0].set_xlabel("log(1 + valor)")

sns.boxplot(
    data=df.assign(classe=df["is_fraud"].map({False: "legítima", True: "fraude"})),
    x="classe",
    y="amount",
    ax=axes[1],
)
axes[1].set_yscale("log")
axes[1].set_title("Valor por classe (escala log)")

plt.tight_layout()
plt.show()

In [ ]:
df["saldo_zerado_origem"] = df["new_balance_org"] <= 0
pd.crosstab(df["saldo_zerado_origem"], df["is_fraud"], normalize="index")

## 5. Série temporal

A coluna `step` representa horas simuladas. A estabilidade temporal do
dataset é o ponto de partida do E1.

In [ ]:
hourly = df.groupby("step").agg(transacoes=("amount", "size"), fraudes=("is_fraud", "sum"))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hourly.index, hourly["transacoes"], color="#2563eb", label="transações")
ax.set_xlabel("step (hora)")
ax.set_ylabel("transações")
ax2 = ax.twinx()
ax2.plot(hourly.index, hourly["fraudes"], color="#dc2626", alpha=0.7, label="fraudes")
ax2.set_ylabel("fraudes")
ax.set_title("Volume e fraudes ao longo do tempo")
plt.show()

## 6. Conclusões do EDA

- Fraude é rara e concentrada em `TRANSFER`/`CASH_OUT`.
- Valores seguem distribuição log-normal; escala log é obrigatória nos gráficos.
- O esvaziamento da conta de origem é o sinal mais forte do padrão histórico.
- Essa concentração é justamente o que torna o modelo vulnerável ao concept
  drift quando surge uma tipologia de fraude que **não** esvazia a conta.